# 03 · Two-stage, anchor-based: Faster R-CNN

**Model.** torchvision Faster R-CNN, ResNet-50-FPN v2, COCO-pretrained; new 9-way box predictor
(8 classes + background).

**Protocol (same for every detector).** Fold 0 (13 held-out clips), every 10th frame (~2.2k training
frames, ~600 validation frames); input long side 1280 (frames are 1280×720 inside the network); AMP;
horizontal flip only; SGD, lr 0.01 for a global batch of 8, momentum 0.9, wd 1e-4, 500-iteration warm-up,
12 epochs with ×0.1 steps at epochs 8 and 11; seed 42; 2× T4 with DDP. Validation: our COCO evaluator on
unmasked test-scale frames, ignore regions as `iscrowd`, 500 detections per image.

| run | change (one at a time) | motivation |
|---|---|---|
| `frcnn_base` | torchvision defaults, 500 detections/image | reference |
| `frcnn_anchors` | anchors 16–256 px instead of 32–512 | 54 % of objects < 32 px at the network input (EDA §3) |
| `frcnn_proposals` | 4000/2000 RPN proposals (train/test) instead of 2000/1000 | up to 180 objects per frame (EDA §4) |
| `frcnn_giou` | GIoU loss on decoded boxes in the RoI head instead of smooth-L1 (loss modification) | IoU-based loss is scale-invariant, smooth-L1 on deltas is not |

Each tweak inherits from `frcnn_anchors` (see `configs/`).

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "ivan-in-a-nutshell/AI_HW_2.git"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Data and the fixed benchmark protocol: fold 0, every 10th frame, unmasked test-scale validation frames
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from visdet.data import find_raw_root, find_prepared, read_prepared, fold_split
from visdet.evaluate import build_gt
from visdet.plotting import setup_style, save, SERIES
from visdet.results import load_log, family_table, best_run, plot_histories, plot_breakdown, styled

RAW, PREP = find_raw_root(), find_prepared()
RUNS = Path("/kaggle/working/runs") if ON_KAGGLE else REPO / "runs"
FOLD, STEP = 0, 10
images, boxes, folds = read_prepared(PREP)
train_rows, val_rows = fold_split(images, folds, FOLD, train_step=STEP, val_step=STEP)
gt = build_gt(val_rows, boxes)
N_GPU = torch.cuda.device_count()
SMOKE = os.environ.get("VISDET_SMOKE") == "1"   # CPU dry run: 1 epoch on a few frames, nothing logged for real
if SMOKE:
    val_rows = val_rows.head(8); gt = build_gt(val_rows, boxes)
setup_style()
print(f"fold {FOLD}: {len(train_rows)} training frames, {len(val_rows)} validation frames; {N_GPU} GPU(s)")

## Training

One subprocess per run (`torchrun` over both GPUs). Each epoch prints the losses and the validation
metrics; the best epoch by mAP is kept. ~1 h of wall time per run on 2× T4.

In [ ]:
# Train every configuration (skips runs that already have a checkpoint)
from visdet.tv_train import launch
CONFIGS = ['frcnn_base', 'frcnn_anchors', 'frcnn_proposals', 'frcnn_giou']
for name in CONFIGS:
    if not (RUNS / name / "best.pt").exists():
        launch(f"configs/{name}.yaml", PREP, RUNS / name, fold=FOLD, train_step=STEP, val_step=STEP,
               **({"epochs": 1, "limit": 8, "workers": 0} if SMOKE else {}))

## Results

In [ ]:
# Validation metrics of every run (ΔmAP against the baseline)
log = load_log()
styled(family_table(log, CONFIGS))

In [ ]:
# Per-class AP and AP by scene density (objects per frame)
cls_cols = [c for c in log.columns if c.startswith("AP_") and c not in ("AP_s", "AP_m", "AP_l") and "dens" not in c]
dens_cols = [c for c in log.columns if "dens" in c]
log.loc[[c for c in CONFIGS if c in log.index], cls_cols + dens_cols].round(3)

In [ ]:
# Validation mAP per epoch
ax = plot_histories([c for c in CONFIGS if c in log.index], title="Faster R-CNN: validation mAP per epoch")
save(ax.figure, "frcnn_curves")

## What each stage contributes

* **Region proposal network (stage 1).** A small conv head slides over every FPN level and, for each
  anchor, predicts class-agnostic *objectness* and a box offset. After NMS it keeps the top-N
  proposals (1000 at test time). It decides **which regions are looked at**: an object without a
  proposal near it cannot be detected later, so the RPN sets the recall ceiling.
* **RoI head (stage 2).** RoIAlign pools a 7×7 feature map per proposal from the matching FPN level.
  An MLP then classifies it into 8 classes + background and regresses a class-specific refinement,
  followed by per-class NMS. It decides **what the region is and exactly where the box goes**, which
  sets precision and localisation (AP at high IoU).

The table measures both stages on 50 validation frames: the recall of the top-N proposals (IoU ≥ 0.5,
class-agnostic) against the recall of the final detections.

In [ ]:
# Recall of RPN proposals vs final detections, by object size
from visdet.config import load_config
from visdet.tv_models import build_model, proposal_recall, TorchvisionPredictor
from visdet.infer import load_frame
BEST = best_run(log, CONFIGS)
cfg = load_config(f"configs/{BEST}.yaml")
model = build_model(cfg); model.load_state_dict(torch.load(RUNS / BEST / "best.pt", map_location="cpu"))
model = model.cuda().eval() if N_GPU else model.eval()
sample = val_rows.sample(min(50, len(val_rows)), random_state=0)
frames = [load_frame(RAW / f, (w, h)) for f, w, h in sample[["file_name", "out_w", "out_h"]].itertuples(index=False)]
objs = [boxes[(boxes.image_id == i) & (boxes.iscrowd == 0)][["x", "y", "w", "h"]].to_numpy() for i in sample.image_id]
print("best run:", BEST)
proposal_recall(model, frames, objs).round(3)

## Failure analysis

Confident detections (score ≥ 0.3) of the best run are labelled with an error type and every object
as found / missed (`visdet.analysis`): background false positives, wrong class, poor localisation,
duplicates, and misses broken down by size and occlusion.

In [ ]:
# Predict the validation frames with the best run and label every error
from visdet.analysis import classify_errors, error_summary, confusion, frame_errors, draw_errors
from visdet.infer import predict_eval_frames, load_frame
predictor = TorchvisionPredictor(model, cfg)
dets = predict_eval_frames(predictor, RAW, val_rows, desc="validation")
det_t, gt_t = classify_errors(boxes[boxes.image_id.isin(val_rows.image_id)], dets, score_thr=0.3)
shares, recall = error_summary(det_t, gt_t)
display(shares.to_frame("share of confident detections").T.round(3))
display(recall.round(3))
confusion(det_t)

In [ ]:
# Three failure examples: most missed objects, most class confusions, most background false positives
fe = frame_errors(det_t, gt_t)
picks = {"missed objects": fe.missed.idxmax(),
          "class confusion": fe.get("wrong_cls", fe.missed).idxmax(),
          "background false positives": fe.get("background", fe.missed).idxmax()}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (what, img_id) in zip(axes, picks.items()):
    r = val_rows.set_index("image_id").loc[img_id]
    frame = load_frame(RAW / r.file_name, (r.out_w, r.out_h))
    vis = draw_errors(frame, gt_t[gt_t.image_id == img_id], det_t[det_t.image_id == img_id])
    ax.imshow(vis[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"{what}: {r['clip']} #{r.frame} ({fe.loc[img_id, 'missed']}/{fe.loc[img_id, 'objects']} missed)", fontsize=8)
fig.suptitle("green = found, yellow = missed, red = background FP, orange = wrong class, magenta = poor localisation", fontsize=8)
fig.tight_layout()
save(fig, "frcnn_failures", photo=True)

## Findings

_Fill in after the runs, using the tables above:_
- **Anchors:** effect on AP_s and on proposal recall for objects < 32 px.
- **Proposals:** whether more proposals raise recall in dense clips (AP by density) and what they cost in FPS.
- **GIoU:** effect on AP75 (localisation) against AP50.
- **Failure cases (≥ 3):** for each example above, what went wrong and the plausible cause
  (e.g. tiny or occluded people below the stride-4 resolution, car ↔ van confusion from a top-down view,
  false positives on painted markings or pedestrians on bikes).

**After the session:** download `/kaggle/working/repo/experiments/` (metrics, per-epoch histories) and
`images/` from the notebook output and commit them, so later notebooks and the report can use them.
Checkpoints stay in the notebook output (`/kaggle/working/runs`).